# CIFAR10 and Transfer Learning

## CIFAR10

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.utils
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import numpy as np
import os

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
train_data = dsets.CIFAR10(root='./data', 
                           train=True,
                           download=True, 
                           transform=transforms.ToTensor())

test_data  = dsets.CIFAR10(root='./data', 
                           train=False,
                           download=True, 
                           transform=transforms.ToTensor())

In [ ]:
batch_size = 128

train_loader = DataLoader(train_data, 
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(test_data, 
                         batch_size=5,
                         shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
def imshow(img, title):
    img = torchvision.utils.make_grid(img, normalize=True)
    npimg = img.numpy()
    fig = plt.figure(figsize = (5, 15))
    plt.imshow(np.transpose(npimg,(1,2,0)))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
train_iter = iter(train_loader)
images, labels = train_iter.next()

imshow(images, "Train Image")

In [ ]:
images.shape

## Train and Evaluate CNN

### Train CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 5),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(64*5*5, 100),
            nn.ReLU(),
            nn.Linear(100, 10)              
        )
        
    def forward(self, x):
        out = self.conv_layer(x)
        out = out.view(-1, 64*5*5)
        out = self.fc_layer(out)
        
        return out
    
model = CNN().cuda()

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
num_epochs = 10

In [ ]:
for epoch in range(num_epochs):

    total_batch = len(train_data) // batch_size
    
    for i, (batch_images, batch_labels) in enumerate(train_loader):
        
        x = batch_images.cuda()
        y = batch_labels.cuda()

        pre = model(x)
        cost = loss(pre, y)

        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        if (i+1) % 200 == 0:
            print('Epoch [%d/%d], lter [%d/%d], Loss: %.4f'
                 %(epoch+1, num_epochs, i+1, total_batch, cost.item()))

### Evalute Model

In [ ]:
correct = 0
total = 0

for images, labels in test_loader:
    
    images = images.cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Accuracy of test images: %f %%' % (100 * float(correct) / total))

In [ ]:
images, labels = iter(test_loader).next()
outputs = model(images.cuda())

_, predicted = torch.max(outputs.data, 1)
    
print('Predicted: ', ' '.join('%5s' % classes[predicted[j]] for j in range(5)))

title = (' '.join('%5s' % classes[labels[j]] for j in range(5)))
imshow(images, title)

## Train and Evaluate Developed CNN

### Preprocessing Data

In [ ]:
transform = transforms.Compose([
    # 데이터 증강
    # Data Augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(32),
    
    # 데이터 정규화
    # Data Normalization
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

# 텐서 이미지를 평균/표준편차로 정규화
# Normalize a tensor image using mean and std
# 채널별 mean=(M1..Mn), std=(S1..Sn) 가 주어졌을 때,
# Given mean=(M1..Mn) and std=(S1..Sn) for n channels,
# 이 변환은 입력 토치 텐서의 각 채널을 정규화함
# This transform normalizes each channel of the input torch tensor
# 입력 텐서를 채널별로 정규화: input[c] = (input[c] - mean[c]) / std[c]
# Normalize per channel: input[c] = (input[c] - mean[c]) / std[c]

train_data = dsets.CIFAR10(root='./data', 
                           train=True,
                           download=True, 
                           transform=transform)

test_data  = dsets.CIFAR10(root='./data', 
                           train=False,
                           download=True, 
                           transform=transform)

In [ ]:
batch_size = 128

train_loader = DataLoader(train_data, 
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(test_data, 
                         batch_size=batch_size,
                         shuffle=False)

### Train CNN

In [ ]:
import torch.nn.init as init

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 5),
            nn.BatchNorm2d(32), # Batch Nomalization
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 5),
            nn.BatchNorm2d(64), # Batch Nomalization
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(64 * 5 * 5, 100),
            nn.ReLU(),
            nn.Dropout(0.1), # Dropout
            nn.Linear(100, 10)              
        )
        
        # 가중치 초기화
        # Weight initialization
        for m in self.modules() :
            if isinstance(m, nn.Conv2d):
                # init.xavier_normal(m.weight.data)
                init.kaiming_normal_(m.weight.data)
                m.bias.data.fill_(0)      
            if isinstance(m, nn.Linear):
                # init.xavier_normal(m.weight.data)
                init.kaiming_normal_(m.weight.data)
                m.bias.data.fill_(0)                
        
    def forward(self, x):
        out = self.conv_layer(x)
        out = out.view(-1, 64*5*5)
        out = self.fc_layer(out)
        
        return out
    
model = CNN().cuda()

### Custom Weight Initialization (Revisited)

In [ ]:
model.conv_layer

In [ ]:
model.conv_layer[0].weight

In [ ]:
model.conv_layer[0].weight.data = torch.ones_like(model.conv_layer[0].weight.data)

In [ ]:
model.conv_layer[0].weight

### Training

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모멘텀 & L2 가중치 규제
# Momentum & L2 weight regularization
# optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-5)

In [ ]:
num_epochs = 10

In [ ]:
for epoch in range(num_epochs):

    total_batch = len(train_data) // batch_size
    
    for i, (batch_images, batch_labels) in enumerate(train_loader):
        
        x = batch_images.cuda()
        y = batch_labels.cuda()

        pre = model(x)
        cost = loss(pre, y)

        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        if (i+1) % 200 == 0:
            print('Epoch [%d/%d], lter [%d/%d], Loss: %.4f'
                 %(epoch+1, num_epochs, i+1, total_batch, cost.item()))

### Warning: BatchNorm and Dropout

In [ ]:
with torch.no_grad():
    for images, labels in test_loader:
        print("First Forward:", model(images[0:3].cuda()))
        print("Second Forward:", model(images[0:3].cuda()))
        print("Thrid Forward:", model(images[0:3].cuda()))
        break

In [ ]:
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        print("First Forward:", model(images[0:3].cuda()))
        print("Second Forward:", model(images[0:3].cuda()))
        print("Thrid Forward:", model(images[0:3].cuda()))
        break

### Eval Developed CNN

In [ ]:
# 이제부터는 반드시 호출해야 함
# From now on, this must be called
# BatchNorm 과 Dropout 은 train/eval 에서 동작이 다름
# BatchNorm and Dropout behave differently in train vs eval mode
model.eval()

# 반대로 다시 학습 모드로 돌리려면
# To switch back to training mode
# model.train()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images.cuda()
        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum()

    print('Accuracy of test images: %f %%' % (100 * float(correct) / total))